# PII Guardrail Model Training

Fine-tune Llama 3.2-1B for PII detection using QLoRA with Unsloth.

**Target Hardware**: Google Colab T4 GPU (16GB VRAM)

**Model**: `unsloth/Llama-3.2-1B-Instruct`

**Technique**: QLoRA (4-bit quantization with LoRA adapters)

## Features
- Detects Indian PII: Aadhaar, PAN
- Detects General PII: Email, Phone, Names, SSN, Credit Cards
- Outputs JSON with entity types, positions, confidence, and reasons


In [ ]:
# Cell 1: Install Unsloth and dependencies
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers scikit-learn


In [ ]:
# Cell 2: Mount Google Drive and setup configuration
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import torch
from datetime import datetime

# Configuration
CONFIG = {
    "model_name": "unsloth/Llama-3.2-1B-Instruct",
    "max_seq_length": 512,
    "load_in_4bit": True,
    "lora_rank": 32,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "batch_size": 4,
    "gradient_accumulation_steps": 4,
    "num_epochs": 3,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.1,
    "output_dir": "/content/drive/MyDrive/pii-guardrail-model",
}

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Cell 3: Load Model with QLoRA (4-bit quantization)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    load_in_4bit=CONFIG["load_in_4bit"],
    dtype=None,  # Auto-detect
)

print(f"Model loaded: {CONFIG['model_name']}")
print(f"Model parameters: {model.num_parameters():,}")


In [ ]:
# Cell 4: Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")


In [ ]:
# Cell 5: Generate ROBUST Synthetic Training Data
# This cell generates realistic, noisy, and adversarial PII examples
# for production-ready model training

import random
import string
import re

random.seed(42)

# ============================================================================
# Character confusion for OCR-style errors
# ============================================================================
CHAR_CONFUSION = {
    '0': ['O', 'o'], '1': ['I', 'l', '|'], '5': ['S', 's'],
    '8': ['B'], 'O': ['0'], 'I': ['1', 'l'], 'S': ['5'], 'B': ['8'],
}

def apply_ocr_noise(text, intensity=0.1):
    """Apply OCR-style character confusion."""
    result = []
    for char in text:
        if random.random() < intensity and char in CHAR_CONFUSION:
            result.append(random.choice(CHAR_CONFUSION[char]))
        else:
            result.append(char)
    return ''.join(result)

def apply_typo(text, intensity=0.1):
    """Apply realistic typos."""
    if len(text) < 3:
        return text
    result = list(text)
    num_errors = max(1, int(len(text) * intensity))
    for _ in range(num_errors):
        pos = random.randint(0, len(result) - 1)
        error_type = random.choice(['transpose', 'delete', 'double'])
        if error_type == 'transpose' and pos < len(result) - 1:
            result[pos], result[pos + 1] = result[pos + 1], result[pos]
        elif error_type == 'delete' and len(result) > 2:
            result.pop(pos)
        elif error_type == 'double':
            result.insert(pos, result[pos])
    return ''.join(result)

# ============================================================================
# PII Generators with augmentation
# ============================================================================
def generate_aadhaar(augment=False):
    first = str(random.randint(2, 9))
    rest = ''.join(random.choices(string.digits, k=11))
    num = first + rest
    
    if not augment:
        return f"{num[:4]} {num[4:8]} {num[8:12]}"
    
    aug_type = random.choice(['clean', 'no_space', 'dashed', 'ocr', 'spaced'])
    if aug_type == 'clean':
        return f"{num[:4]} {num[4:8]} {num[8:12]}"
    elif aug_type == 'no_space':
        return num
    elif aug_type == 'dashed':
        return f"{num[:4]}-{num[4:8]}-{num[8:12]}"
    elif aug_type == 'ocr':
        return apply_ocr_noise(f"{num[:4]} {num[4:8]} {num[8:12]}", 0.15)
    else:  # spaced
        return ' '.join(num)

def generate_pan(augment=False):
    letters = ''.join(random.choices(string.ascii_uppercase, k=5))
    digits = ''.join(random.choices(string.digits, k=4))
    check = random.choice(string.ascii_uppercase)
    pan = f"{letters}{digits}{check}"
    
    if not augment:
        return pan
    
    aug_type = random.choice(['clean', 'lower', 'mixed', 'ocr', 'spaced'])
    if aug_type == 'clean':
        return pan
    elif aug_type == 'lower':
        return pan.lower()
    elif aug_type == 'mixed':
        return ''.join(c.upper() if random.random() < 0.5 else c.lower() for c in pan)
    elif aug_type == 'ocr':
        return apply_ocr_noise(pan, 0.2)
    else:
        return f"{pan[:5]} {pan[5:9]} {pan[9]}"

def generate_email(augment=False):
    names = ['rahul', 'priya', 'amit', 'neha', 'darshan', 'krishna']
    domains = ['gmail.com', 'yahoo.com', 'outlook.com', 'company.in']
    email = f"{random.choice(names)}{random.randint(1,99)}@{random.choice(domains)}"
    
    if not augment:
        return email
    
    aug_type = random.choice(['clean', 'upper', 'at_spaced', 'at_written', 'typo'])
    if aug_type == 'clean':
        return email
    elif aug_type == 'upper':
        return email.upper()
    elif aug_type == 'at_spaced':
        return email.replace('@', ' @ ')
    elif aug_type == 'at_written':
        return email.replace('@', ' at ')
    else:
        parts = email.split('@')
        return f"{apply_typo(parts[0], 0.15)}@{parts[1]}"

def generate_phone(augment=False):
    first = random.choice(['6', '7', '8', '9'])
    rest = ''.join(random.choices(string.digits, k=9))
    num = first + rest
    
    if not augment:
        return f"+91 {num[:5]} {num[5:]}"
    
    aug_type = random.choice(['clean', 'no_code', 'dashed', 'ocr', 'spaced'])
    if aug_type == 'clean':
        return f"+91 {num[:5]} {num[5:]}"
    elif aug_type == 'no_code':
        return num
    elif aug_type == 'dashed':
        return f"{num[:3]}-{num[3:6]}-{num[6:]}"
    elif aug_type == 'ocr':
        return apply_ocr_noise(f"+91-{num}", 0.1)
    else:
        return ' '.join(num)

def generate_name():
    first = random.choice(['Rahul', 'Priya', 'Amit', 'Neha', 'Darshan', 'Krishna'])
    last = random.choice(['Sharma', 'Patel', 'Singh', 'Kumar', 'Gupta', 'Reddy'])
    return f"{first} {last}"

def generate_ssn():
    area = random.randint(100, 899)
    while area == 666:
        area = random.randint(100, 899)
    group = random.randint(10, 99)
    serial = random.randint(1000, 9999)
    return f"{area}-{group}-{serial}"

# ============================================================================
# Template Categories
# ============================================================================
CLEAN_TEMPLATES = [
    ("My Aadhaar number is {value}.", "IN_AADHAAR", lambda: generate_aadhaar(False)),
    ("PAN: {value}", "IN_PAN", lambda: generate_pan(False)),
    ("Email: {value}", "EMAIL_ADDRESS", lambda: generate_email(False)),
    ("Mobile: {value}", "PHONE_NUMBER", lambda: generate_phone(False)),
    ("My name is {value}.", "PERSON", generate_name),
    ("SSN: {value}", "US_SSN", generate_ssn),
]

NOISY_TEMPLATES = [
    ("aadhaar no {value} pls verify", "IN_AADHAAR", lambda: generate_aadhaar(True)),
    ("pan card {value} for kyc", "IN_PAN", lambda: generate_pan(True)),
    ("mail me @ {value} asap", "EMAIL_ADDRESS", lambda: generate_email(True)),
    ("call {value} urgent", "PHONE_NUMBER", lambda: generate_phone(True)),
    ("mera aadhaar hai {value}", "IN_AADHAAR", lambda: generate_aadhaar(True)),
    ("mera pan {value} hai", "IN_PAN", lambda: generate_pan(True)),
]

ADVERSARIAL_TEMPLATES = [
    ("a a d h a a r: {value}", "IN_AADHAAR", lambda: generate_aadhaar(True)),
    ("p a n number: {value}", "IN_PAN", lambda: generate_pan(True)),
    ("e m a i l: {value}", "EMAIL_ADDRESS", lambda: generate_email(True)),
    ("[HIDDEN] {value} [/HIDDEN]", "IN_AADHAAR", lambda: generate_aadhaar(True)),
]

# Clean negatives
NEGATIVE_TEMPLATES = [
    "The weather today is sunny with a high of 25 degrees.",
    "Please submit the report by Friday.",
    "The meeting is scheduled for 3 PM tomorrow.",
    "Thank you for your patience.",
    "The project deadline has been extended.",
]

# DANGEROUS NEGATIVES (look like PII but aren't!)
DANGEROUS_NEGATIVES = [
    "Order ID: 0123 4567 8901",  # Starts with 0, not Aadhaar
    "Transaction: 1234 5678 9012",  # Starts with 1
    "Product: ABCDE12345",  # Wrong PAN format
    "SKU: XYZDE9999X",  # Invalid 4th char
    "Account: 0123456789",  # Starts with 0, not phone
    "Policy: 1234567890",  # Starts with 1
    "Code: 000-00-0000",  # Invalid SSN
    "Format: 666-12-3456",  # Invalid SSN (666)
    "Use format: user@domain",  # Not real email
    "Pattern: name@company.com",  # Example text
    "Reference: 0000 0000 0000",  # All zeros
    "Version 1.2.3.4",  # Version number
    "Server: 192.168.1.100",  # IP address
    "Promo: SAVE20NOW",  # Promo code
]

SYSTEM_PROMPT = """You are a PII detection model. Analyze text and identify PII entities with their exact positions.

Output JSON with: flagged (bool), entities (array with type, value, start, end), confidence (0-1), reason (string).

Entity types: IN_AADHAAR, IN_PAN, EMAIL_ADDRESS, PHONE_NUMBER, PERSON, US_SSN, CREDIT_CARD

Important:
- Report confidence based on how clear/corrupted the PII appears
- Handle typos, OCR errors, and obfuscation attempts
- Don't flag non-PII identifiers like order IDs, product codes, or dates"""

def generate_sample(difficulty='mixed'):
    """Generate a sample with specified difficulty."""
    
    # Distribution: 25% easy, 25% medium, 15% hard, 10% adversarial, 10% negative, 15% dangerous_neg
    if difficulty == 'mixed':
        r = random.random()
        if r < 0.25:
            difficulty = 'easy'
        elif r < 0.50:
            difficulty = 'medium'
        elif r < 0.65:
            difficulty = 'hard'
        elif r < 0.75:
            difficulty = 'adversarial'
        elif r < 0.85:
            difficulty = 'negative'
        else:
            difficulty = 'dangerous_negative'
    
    if difficulty == 'negative':
        text = random.choice(NEGATIVE_TEMPLATES)
        return {
            "conversations": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f'Detect PII in: "{text}"'},
                {"role": "assistant", "content": json.dumps({
                    "flagged": False, "entities": [], "confidence": 0.95,
                    "reason": "No PII detected"
                }, indent=2)}
            ]
        }
    
    if difficulty == 'dangerous_negative':
        text = random.choice(DANGEROUS_NEGATIVES)
        return {
            "conversations": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f'Detect PII in: "{text}"'},
                {"role": "assistant", "content": json.dumps({
                    "flagged": False, "entities": [], "confidence": 0.90,
                    "reason": "No PII detected - identifier is not sensitive personal information"
                }, indent=2)}
            ]
        }
    
    # Select template based on difficulty
    if difficulty == 'easy':
        template, entity_type, generator = random.choice(CLEAN_TEMPLATES)
        confidence = 0.95
    elif difficulty == 'medium':
        template, entity_type, generator = random.choice(NOISY_TEMPLATES)
        confidence = 0.80
    elif difficulty == 'hard':
        template, entity_type, generator = random.choice(NOISY_TEMPLATES)
        confidence = 0.65
    else:  # adversarial
        template, entity_type, generator = random.choice(ADVERSARIAL_TEMPLATES)
        confidence = 0.55
    
    value = generator()
    text = template.replace("{value}", value)
    start = text.find(value)
    end = start + len(value)
    
    readable_type = entity_type.replace("_", " ").replace("IN ", "").title()
    
    if confidence < 0.7:
        reason = f"Possibly detected 1 {readable_type} (low confidence due to noise)"
    else:
        reason = f"Detected 1 {readable_type}"
    
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f'Detect PII in: "{text}"'},
            {"role": "assistant", "content": json.dumps({
                "flagged": True,
                "entities": [{"type": entity_type, "value": value, "start": start, "end": end}],
                "confidence": confidence,
                "reason": reason
            }, indent=2)}
        ]
    }

# Generate ROBUST dataset
NUM_SAMPLES = 5000
train_data = [generate_sample('mixed') for _ in range(int(NUM_SAMPLES * 0.8))]
eval_data = [generate_sample('mixed') for _ in range(int(NUM_SAMPLES * 0.2))]

print(f"Generated {len(train_data)} training samples")
print(f"Generated {len(eval_data)} evaluation samples")

# Count distribution
difficulty_counts = {'positive': 0, 'negative': 0, 'dangerous_neg': 0}
for item in train_data:
    output = json.loads(item['conversations'][2]['content'])
    if output['flagged']:
        difficulty_counts['positive'] += 1
    elif 'not sensitive' in output['reason']:
        difficulty_counts['dangerous_neg'] += 1
    else:
        difficulty_counts['negative'] += 1

print(f"\nDistribution: {difficulty_counts}")

# Save to files
with open('train.jsonl', 'w') as f:
    for item in train_data:
        f.write(json.dumps(item) + '\n')

with open('eval.jsonl', 'w') as f:
    for item in eval_data:
        f.write(json.dumps(item) + '\n')

print("\nSample (easy):")
print(json.dumps(generate_sample('easy'), indent=2))
print("\nSample (adversarial):")
print(json.dumps(generate_sample('adversarial'), indent=2))
print("\nSample (dangerous_negative):")
print(json.dumps(generate_sample('dangerous_negative'), indent=2))


In [ ]:
# Cell 6: Load and format datasets
from datasets import load_dataset

train_dataset = load_dataset('json', data_files='train.jsonl', split='train')
eval_dataset = load_dataset('json', data_files='eval.jsonl', split='train')

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Eval dataset: {len(eval_dataset)} samples")

def format_chat(example):
    """Format conversations into the Llama chat template."""
    formatted = tokenizer.apply_chat_template(
        example['conversations'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": formatted}

train_dataset = train_dataset.map(format_chat)
eval_dataset = eval_dataset.map(format_chat)

print("\nFormatted sample preview:")
print(train_dataset[0]['text'][:500])


In [ ]:
# Cell 7: Setup Training
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    args=training_args,
)

print("Trainer initialized!")
print(f"Training for {CONFIG['num_epochs']} epochs")
print(f"Effective batch size: {CONFIG['batch_size'] * CONFIG['gradient_accumulation_steps']}")


In [ ]:
# Cell 8: Train the model
print(f"Starting training at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

trainer_stats = trainer.train()

print(f"\nTraining completed at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Training loss: {trainer_stats.training_loss:.4f}")


In [ ]:
# Cell 9: Evaluation - Calculate Precision, Recall, F1 per entity type
from sklearn.metrics import precision_recall_fscore_support
from collections import defaultdict

def extract_entities_from_output(output_text):
    """Extract entities from model output JSON."""
    try:
        start = output_text.find('{')
        end = output_text.rfind('}') + 1
        if start >= 0 and end > start:
            data = json.loads(output_text[start:end])
            return data.get('entities', []), data.get('flagged', False)
    except:
        pass
    return [], False

def evaluate_model(model, tokenizer, eval_samples, max_samples=100):
    """Evaluate model on eval set."""
    FastLanguageModel.for_inference(model)
    
    results = {
        'true_flagged': [],
        'pred_flagged': [],
        'true_entities': [],
        'pred_entities': [],
    }
    
    for i, sample in enumerate(eval_samples[:max_samples]):
        if i % 20 == 0:
            print(f"Evaluating sample {i+1}/{max_samples}...")
        
        conversations = sample['conversations']
        ground_truth = json.loads(conversations[2]['content'])
        
        messages = conversations[:2]
        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        output_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        pred_entities, pred_flagged = extract_entities_from_output(output_text)
        
        results['true_flagged'].append(ground_truth['flagged'])
        results['pred_flagged'].append(pred_flagged)
        results['true_entities'].append(ground_truth['entities'])
        results['pred_entities'].append(pred_entities)
    
    return results

print("Running evaluation...")
eval_results = evaluate_model(model, tokenizer, eval_data, max_samples=100)


In [ ]:
# Cell 10: Print Precision-Recall Report
precision, recall, f1, _ = precision_recall_fscore_support(
    eval_results['true_flagged'],
    eval_results['pred_flagged'],
    average='binary'
)

print("=" * 60)
print("PII DETECTION METRICS - PRECISION/RECALL REPORT")
print("=" * 60)
print(f"\nOverall Flagged Detection:")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1 Score:  {f1:.4f}")

# Per-entity-type metrics
entity_metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

for true_ents, pred_ents in zip(eval_results['true_entities'], eval_results['pred_entities']):
    true_set = {(e['type'], e['value']) for e in true_ents}
    pred_set = {(e['type'], e['value']) for e in pred_ents}
    
    for etype, value in true_set:
        if (etype, value) in pred_set:
            entity_metrics[etype]['tp'] += 1
        else:
            entity_metrics[etype]['fn'] += 1
    
    for etype, value in pred_set:
        if (etype, value) not in true_set:
            entity_metrics[etype]['fp'] += 1

print(f"\nPer-Entity-Type Metrics:")
print("-" * 60)
print(f"{'Entity Type':<20} {'Precision':>12} {'Recall':>12} {'F1':>12}")
print("-" * 60)

for etype in sorted(entity_metrics.keys()):
    counts = entity_metrics[etype]
    tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    print(f"{etype:<20} {prec:>12.4f} {rec:>12.4f} {f1_score:>12.4f}")

print("=" * 60)


In [ ]:
# Cell 11: Save LoRA adapters
lora_path = f"{CONFIG['output_dir']}/lora-adapters"
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)

print(f"LoRA adapters saved to: {lora_path}")

# Save training config and metrics
metrics = {
    "config": CONFIG,
    "training_loss": trainer_stats.training_loss,
    "evaluation": {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    },
    "entity_metrics": {k: dict(v) for k, v in entity_metrics.items()},
    "timestamp": datetime.now().isoformat(),
}

metrics_path = f"{CONFIG['output_dir']}/training_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics saved to: {metrics_path}")


In [ ]:
# Cell 12: Test Inference with sample inputs
FastLanguageModel.for_inference(model)

test_texts = [
    "My Aadhaar number is 2345 6789 0123.",
    "Contact me at rahul.sharma@gmail.com or call +91 98765 43210.",
    "My PAN card number is ABCPD1234E for tax filing.",
    "The weather today is sunny with a high of 25 degrees.",
    "User Darshan Krishna with Aadhaar 9876 5432 1098 and PAN BNZPM2501F registered.",
]

print("=" * 70)
print("TEST INFERENCE RESULTS")
print("=" * 70)

for text in test_texts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Detect PII in: "{text}"'}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    output_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f"\nInput: {text}")
    print(f"Output:\n{output_text}")
    print("-" * 70)


In [ ]:
# Cell 13: Export for deployment - zip LoRA adapters
import shutil

zip_path = "/content/pii-guardrail-lora"
shutil.make_archive(zip_path, 'zip', lora_path)

print(f"Created: {zip_path}.zip")
print("\nTo download, uncomment and run:")
print("# from google.colab import files")
print(f"# files.download('{zip_path}.zip')")
